In [1]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
from langsmith import Client

RESULTS_DIR = Path("../data/langsmith/results")
SPLITS = ["id", "ood"]
SCENARIOS = [1, 2, 3]

DROP_COLS = ["id", "session_name", "repetition", "status", "error", "latency", "tokens", "total_cost"]

EVAL_COLS = ["agent_goal_accuracy", "context_relevance", "disease_accuracy", "faithfulness", "trajectory_accuracy"]
COMMENT_COLS = [f"{col}_comment" for col in EVAL_COLS]

ANSWER_MAX_LEN = 300
COMMENT_MAX_LEN = 300


def parse_row(row):
    inputs = json.loads(row["inputs"]) if isinstance(row["inputs"], str) else row["inputs"]
    ref = json.loads(row["reference_outputs"]) if isinstance(row["reference_outputs"], str) else row["reference_outputs"]
    out = json.loads(row["outputs"]) if isinstance(row["outputs"], str) else row["outputs"]
    run = json.loads(row["run"]) if isinstance(row["run"], str) else row["run"]

    meta = ref.get("metadata", {})

    tool_calls = ref.get("reference_tool_calls", [])
    tool_names = [t.get("name") for t in tool_calls] if tool_calls else []

    answer = out.get("final_answer") or ""
    if len(answer) > ANSWER_MAX_LEN:
        answer = answer[:ANSWER_MAX_LEN] + "..."

    return {
        "run_id": run.get("id"),
        "image_url": inputs.get("image_url"),
        "plant": meta.get("plant"),
        "true_class": meta.get("class"),
        "pathogen_type": meta.get("pathogen_type"),
        "prompt_type": meta.get("prompt_type"),
        "user_text": inputs.get("user_text"),
        "reference_goal": ref.get("reference_goal"),
        "reference_tool_calls": tool_names,
        "final_answer": answer,
    }


def fetch_feedback_comments(run_ids, client=None):
    client = client or Client()
    comments_map = defaultdict(dict)

    for fb in client.list_feedback(run_ids=run_ids):
        if fb.run_id is None:
            continue
        comments_map[str(fb.run_id)][f"{fb.key}_comment"] = fb.comment

    return comments_map


def attach_feedback_comments(df, client=None):
    run_ids = [str(run_id) for run_id in df["run_id"].dropna().unique().tolist()]
    comments_map = fetch_feedback_comments(run_ids, client=client)
    enriched = df.copy()

    for comment_col in COMMENT_COLS:
        enriched[comment_col] = enriched["run_id"].map(
            lambda run_id: (
                (val[:COMMENT_MAX_LEN] + "...") if (val := comments_map.get(str(run_id), {}).get(comment_col)) and len(val) > COMMENT_MAX_LEN else val
            ) if pd.notna(run_id) else None
        )

    return enriched


dfs = {}
for split in SPLITS:
    for scenario in SCENARIOS:
        split_dir = RESULTS_DIR / split
        csv_file = next(split_dir.glob(f"*scenario{scenario}*.csv"))
        key = f"{split}_s{scenario}"
        df = pd.read_csv(csv_file)
        df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
        parsed = df.apply(parse_row, axis=1, result_type="expand")
        dfs[key] = pd.concat([parsed, df[EVAL_COLS]], axis=1)
        print(f"{key}: {len(dfs[key])} rows — {csv_file.name}")


id_s1: 89 rows — experiment-vqa-id-scenario1-20260125-0958-f4820b41.csv
id_s2: 89 rows — experiment-vqa-id-scenario2-20260125-0649-2e8f40b2.csv
id_s3: 89 rows — experiment-vqa-id-scenario3-20260127-0207-d518afee.csv
ood_s1: 40 rows — experiment-vqa-ood-scenario1-20260125-1439-7edf7bc0.csv
ood_s2: 40 rows — experiment-vqa-ood-scenario2-20260129-0809-ed883d77.csv
ood_s3: 40 rows — experiment-vqa-ood-scenario3-20260125-1947-2adfbc3f.csv


In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

client = Client()

rename_map = {
    "run_id": "Run ID",
    "image_url": "URL Citra",
    "plant": "Tanaman",
    "true_class": "Kelas Sebenarnya",
    "pathogen_type": "Tipe Patogen",
    "prompt_type": "Skenario",
    "user_text": "Teks Input",
    "reference_goal": "Tujuan Referensi",
    "reference_tool_calls": "Panggilan Alat Referensi",
    "final_answer": "Jawaban Akhir",
    "agent_goal_accuracy": "Agent Goal Accuracy",
    "context_relevance": "Context Relevance",
    "disease_accuracy": "Disease Accuracy",
    "faithfulness": "Faithfulness",
    "trajectory_accuracy": "Trajectory Accuracy",
    "agent_goal_accuracy_comment": "Agent Goal Accuracy Comment",
    "context_relevance_comment": "Context Relevance Comment",
    "disease_accuracy_comment": "Disease Accuracy Comment",
    "faithfulness_comment": "Faithfulness Comment",
    "trajectory_accuracy_comment": "Trajectory Accuracy Comment"
}

id_dfs = []
ood_dfs = []

for split in ["id", "ood"]:
    for scenario in [1, 2, 3]:
        key = f"{split}_s{scenario}"
        if key in dfs:
            print(f"Processing {key}...")
            # Attach feedback comments
            df = attach_feedback_comments(dfs[key], client=client)
            
            # Rename columns
            df = df.rename(columns=rename_map)
            
            # Set Scenario column
            df["Skenario"] = f"{scenario}"
            
            if split == "id":
                id_dfs.append(df)
            else:
                ood_dfs.append(df)

# Concatenate all dataframes
df_id = pd.concat(id_dfs, ignore_index=True)
df_ood = pd.concat(ood_dfs, ignore_index=True)

print(f"ID DataFrame shape: {df_id.shape}")
print(f"OOD DataFrame shape: {df_ood.shape}")
df_id.head()

Processing id_s1...
Processing id_s2...
Processing id_s3...
Processing ood_s1...
Processing ood_s2...
Processing ood_s3...
ID DataFrame shape: (267, 20)
OOD DataFrame shape: (120, 20)


,Run ID,URL Citra,Tanaman,Kelas Sebenarnya,Tipe Patogen,Skenario,Teks Input,Tujuan Referensi,Panggilan Alat Referensi,Jawaban Akhir,Agent Goal Accuracy,Context Relevance,Disease Accuracy,Faithfulness,Trajectory Accuracy,Agent Goal Accuracy Comment,Context Relevance Comment,Disease Accuracy Comment,Faithfulness Comment,Trajectory Accuracy Comment
0,019bf316-90c3-7ad1-b501-84564146aef9,https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0249_rice_leaf.jpg,rice,rice leaf,healthy,1,Tanaman padi saya ini terlihat sangat hijau dan segar di sawah. Apakah ini tandanya tanaman saya sehat? Ada tips untuk perawatannya agar tetap subur?,"Nilai status kesehatan rice dari gambar, konfirmasi bahwa tanaman sehat, dan berikan rekomendasi perawatan dan pemeliharaan.",[knowledgebase_search],"Dari foto yang Anda bagikan, tanaman padi Anda memang terlihat tumbuh sangat rimbun dan hijau pekat. Namun, jika diperhatikan lebih dekat pada bagian depan dan tengah foto, terdapat beberapa gejala yang perlu diwaspadai agar tanaman tetap sehat hingga panen.\n\n**Hasil Pengamatan Visual**\nMeskipun sec...",1.0,1.0,1.0,1.0,1.0,"The agent fully achieves the reference goal by providing a comprehensive and nuanced assessment. It correctly analyzes the image, noting not just the overall greenness but also the subtle signs of potential issues like yellowing tips and spots. The agent effectively uses the `plant_disease_identific...",None,"The score is 1.0 because the BinaryJudgementNode at Level 0 correctly identified the plant as healthy, as the agent's overall conclusion was that the plant was in good condition despite mentioning potential early signs, which aligns with the criteria for a healthy plant case. This positive judgment ...","The response accurately identifies potential diseases, Hawar Daun Bakteri (Bacterial Leaf Blight) and Blas (Blast), which are directly supported by the retrieval context and tool outputs. The visual observations of yellowing tips and spots are reasonable given the image. All care recommendations, su...","Trajektori referensi mengharuskan pemanggilan `knowledgebase_search`. Trajektori aktual tidak hanya memenuhi persyaratan ini dengan memanggil `knowledgebase_search` beberapa kali, tetapi juga menambahkan alat-alat yang sangat relevan dan diperlukan untuk menjawab permintaan pengguna secara lengkap. ..."
1,019bf316-90c5-77f1-9dc6-d5d374d2b15a,https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0144_squash_powdery_mildew.jpg,squash,squash powdery mildew,fungal,1,Squash saya memiliki bercak-bercak putih seperti bedak di daunnya. Ini kira-kira kenapa ya dan bagaimana cara mengobatinya?,"Lakukan triase visual pada gejala squash yang terlihat di gambar, gunakan alat identifikasi penyakit untuk mengonfirmasi squash powdery mildew, lalu berikan panduan manajemen dan pengobatan berbasis bukti.","[closed_set_leaf_detection, plant_disease_identification, knowledgebase_search]","Berdasarkan foto yang Anda unggah, daun squash tersebut menunjukkan gejala klasik **Embun Tepung (Powdery Mildew)**.\n\n**Hasil Observasi Visual:**\n* **Gejala:** Terlihat bercak-bercak putih yang menyerupai bedak atau tepung pada permukaan atas daun.\n* **Penyebaran:** Bercak-bercak tersebut mulai ...",1.0,1.0,1.0,1.0,1.0,"The agent fully achieved the reference goal by correctly performing all three required tasks: visual analysis, disease identification, and providing management guidance. It accurately used the `plant_disease_identification` tool to confirm the issue as squash powdery mildew, consistent with the visu...",None,"The score is 1.0 because the BinaryJudgementNode at Level 0 correctly identified it as a disease case, leading to the BinaryJudgementNode at Level 1, which confirmed the agent's accurate identification of 'squash powdery mildew', resulting in a final True verdict from the VerdictNode at Level 2.","The model correctly identifies the d

In [3]:
def compact_dataframe(df):
    compact_df = pd.DataFrame()
    
    if "Skenario" in df.columns:
        compact_df["Skenario"] = df["Skenario"]
    if "URL Citra" in df.columns:
        compact_df["URL Citra"] = df["URL Citra"]
        
    compact_df["Info Penyakit"] = df.apply(lambda row: {
        "Tanaman": row["Tanaman"],
        "Kelas": row["Kelas Sebenarnya"],
        "Patogen": row["Tipe Patogen"]
    }, axis=1)
    
    compact_df["Percakapan"] = df.apply(lambda row: {
        "Input": row["Teks Input"],
        "Jawaban": row["Jawaban Akhir"]
    }, axis=1)
    
    compact_df["Referensi"] = df.apply(lambda row: {
        "Tujuan": row.get("Tujuan Referensi"),
        "Panggilan Alat": row.get("Panggilan Alat Referensi")
    }, axis=1)
        
    metrics = [
        "Agent Goal Accuracy",
        "Context Relevance",
        "Disease Accuracy",
        "Faithfulness",
        "Trajectory Accuracy"
    ]
    for metric in metrics:
        compact_df[metric] = df.apply(lambda row: {
            "score": row[metric],
            "comment": row[f"{metric} Comment"]
        }, axis=1)
        
    return compact_df

df_id = compact_dataframe(df_id)
df_ood = compact_dataframe(df_ood)

print(f"ID DataFrame shape: {df_id.shape}")
print(f"OOD DataFrame shape: {df_ood.shape}")
df_id.head()

ID DataFrame shape: (267, 10)
OOD DataFrame shape: (120, 10)


,Skenario,URL Citra,Info Penyakit,Percakapan,Referensi,Agent Goal Accuracy,Context Relevance,Disease Accuracy,Faithfulness,Trajectory Accuracy
0,1,https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0249_rice_leaf.jpg,"{'Tanaman': 'rice', 'Kelas': 'rice leaf', 'Patogen': 'healthy'}","{'Input': 'Tanaman padi saya ini terlihat sangat hijau dan segar di sawah. Apakah ini tandanya tanaman saya sehat? Ada tips untuk perawatannya agar tetap subur?', 'Jawaban': 'Dari foto yang Anda bagikan, tanaman padi Anda memang terlihat tumbuh sangat rimbun dan hijau pekat. Namun, jika diperhatikan lebih dekat pada bagian depan dan tengah foto, terdapat beberapa gejala yang perlu diwaspadai agar tanaman tetap sehat hingga panen. **Hasil Pengamatan Visual** Meskipun sec...'}","{'Tujuan': 'Nilai status kesehatan rice dari gambar, konfirmasi bahwa tanaman sehat, dan berikan rekomendasi perawatan dan pemeliharaan.', 'Panggilan Alat': ['knowledgebase_search']}","{'score': 1.0, 'comment': 'The agent fully achieves the reference goal by providing a comprehensive and nuanced assessment. It correctly analyzes the image, noting not just the overall greenness but also the subtle signs of potential issues like yellowing tips and spots. The agent effectively uses the `plant_disease_identific...'}","{'score': 1.0, 'comment': None}","{'score': 1.0, 'comment': 'The score is 1.0 because the BinaryJudgementNode at Level 0 correctly identified the plant as healthy, as the agent's overall conclusion was that the plant was in good condition despite mentioning potential early signs, which aligns with the criteria for a healthy plant case. This positive judgment ...'}","{'score': 1.0, 'comment': 'The response accurately identifies potential diseases, Hawar Daun Bakteri (Bacterial Leaf Blight) and Blas (Blast), which are directly supported by the retrieval context and tool outputs. The visual observations of yellowing tips and spots are reasonable given the image. All care recommendations, su...'}","{'score': 1.0, 'comment': 'Trajektori referensi mengharuskan pemanggilan `knowledgebase_search`. Trajektori aktual tidak hanya memenuhi persyaratan ini dengan memanggil `knowledgebase_search` beberapa kali, tetapi juga menambahkan alat-alat yang sangat relevan dan diperlukan untuk menjawab permintaan pengguna secara lengkap. ...'}"
1,1,https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0144_squash_powdery_mildew.jpg,"{'Tanaman': 'squash', 'Kelas': 'squash powdery mildew', 'Patogen': 'fungal'}","{'Input': 'Squash saya memiliki bercak-bercak putih seperti bedak di daunnya. Ini kira-kira kenapa ya dan bagaimana cara mengobatinya?', 'Jawaban': 'Berdasarkan foto yang Anda unggah, daun squash tersebut menunjukkan gejala klasik **Embun Tepung (Powdery Mildew)**. **Hasil Observasi Visual:** * **Gejala:** Terlihat bercak-bercak putih yang menyerupai bedak atau tepung pada permukaan atas daun. * **Penyebaran:** Bercak-bercak tersebut mulai ...'}","{'Tujuan': 'Lakukan triase visual pada gejala squash yang terlihat di gambar, gunakan alat identifikasi penyakit untuk mengonfirmasi squash powdery mildew, lalu berikan panduan manajemen dan pengobatan berbasis bukti.', 'Panggilan Alat': ['closed_set_leaf_detection', 'plant_disease_identification', 'knowledgebase_search']}","{'score': 1.0, 'comment': 'The agent fully achieved the reference goal by correctly performing all three required tasks: visual analysis, disease identification, and providing management guidance. It accurately used the `plant_disease_identification` tool to confirm the issue as squash powdery mildew, consistent with the visu...'}","{'score': 1.0, 'comment': None}","{'score': 1.0, 'comment': 'The score is 1.0 because the BinaryJudgementNode at Level 0 correctly identified it as a disease case, leading to the BinaryJudgementNode at Level 1, which confirmed the agent's accurate identification of 'squash powdery 

In [4]:
df_ood.head()

,Skenario,URL Citra,Info Penyakit,Percakapan,Referensi,Agent Goal Accuracy,Context Relevance,Disease Accuracy,Faithfulness,Trajectory Accuracy
0,1,https://thesis-assets.andyathsid.com/ood/evaluation/images/000_radish_flea_beetle.jpg,"{'Tanaman': 'radish', 'Kelas': 'Radish Flea Beetle', 'Patogen': 'pest'}","{'Input': 'Radish saya daunnya bolong-bolong seperti dimakan. Ini kira-kira kenapa ya dan bagaimana cara mengobatinya?', 'Jawaban': 'Berdasarkan foto yang Anda berikan, daun radish (lobak) tersebut menunjukkan tanda-tanda kerusakan akibat **hama pemakan daun**, bukan penyakit yang disebabkan oleh jamur atau bakteri. **Hasil Observasi Visual:** * Terdapat lubang-lubang dengan ukuran bervariasi dan bentuk yang tidak beraturan pa...'}","{'Tujuan': 'Lakukan triase visual pada gejala radish yang terlihat di gambar, gunakan alat identifikasi penyakit untuk mengonfirmasi Pest (Flea beetles - Phyllotreta spp.), lalu berikan panduan manajemen dan pengobatan berbasis bukti.', 'Panggilan Alat': ['plant_disease_identification', 'knowledgebase_search']}","{'score': 0.8, 'comment': 'Agen berhasil mengidentifikasi masalah umum sebagai kerusakan akibat hama dan memberikan panduan manajemen yang komprehensif. Namun, agen gagal mencapai tujuan referensi spesifik untuk mengidentifikasi 'Flea beetles'. Alat `plant_disease_identification` tidak memberikan hasil yang relevan. Meskipun ...'}","{'score': 1.0, 'comment': None}","{'score': 0.8, 'comment': 'The score is 0.8 because the agent failed to definitively identify 'Radish Flea Beetle' as the main pest (BinaryJudgementNode Level 0 and Level 1 verdicts were False). However, the BinaryJudgementNode Level 2 returned 'True' as 'Radish Flea Beetle' was mentioned in the differential diagnosis (top 2-...'}","{'score': 1.0, 'comment': 'The response accurately identifies the visual evidence of pest damage and correctly dismisses the disease information from the retrieval context. The specific claims about potential pests, particularly slugs/snails, are directly supported by the web search results. The recommendations provided are s...'}","{'score': 1.0, 'comment': 'Trajektori referensi mengharuskan dua alat wajib: `plant_disease_identification` dan `knowledgebase_search`. Trajektori aktual berhasil memanggil kedua alat ini. Faktanya, trajektori aktual jauh lebih komprehensif dan menunjukkan penalaran yang kuat dalam menghadapi ketidakpastian. Pertama, agen sec...'}"
1,1,https://thesis-assets.andyathsid.com/ood/evaluation/images/000_radish_mosaic_virus.jpg,"{'Tanaman': 'radish', 'Kelas': 'Radish Mosaic Virus', 'Patogen': 'viral'}","{'Input': 'Radish saya memiliki pola seperti mosaik pada daunnya dengan area hijau terang dan gelap, dan daunnya juga terlihat agak aneh dan pertumbuhannya terhambat. Ini kira-kira kenapa ya dan bagaimana cara mengobatinya?', 'Jawaban': 'Berdasarkan foto dan deskripsi yang Anda berikan, tanaman radish (lobak) Anda tampaknya mengalami dua masalah sekaligus: serangan hama penggorok daun dan kemungkinan infeksi virus. **Analisis Visual dan Gejala** Dari foto yang Anda unggah, saya melihat adanya bercak-bercak lebar berwarna cokelat mu...'}","{'Tujuan': 'Lakukan triase visual pada gejala radish yang terlihat di gambar, gunakan alat identifikasi penyakit untuk mengonfirmasi Viral (Turnip mosaic virus), lalu berikan panduan manajemen dan pengobatan berbasis bukti.', 'Panggilan Alat': ['plant_disease_identification', 'knowledgebase_search']}","{'score': 0.6, 'comment': 'Agen secara akurat mengidentifikasi dua masalah: kemungkinan infeksi virus (sesuai dengan tujuan referensi) dan serangan penggorok daun berdasarkan analisis visual gambar. Namun, diagnosis inti dari Turnip Mosaic Virus tidak didukung oleh output dari alat `plant_disease_identification` atau `knowled...'}","{'score': 1.0, 'comment': None}","{'score': 1.0, 'comment': 'The score is 1.0 because the DAG traversal first confirmed it was a disease case (Level 0 Verdict: False), then determined that the agent's identificatio

In [5]:
# Tampilkan 5 baris pertama per skenario sebagai string LaTeX dan CSV
df_id_sample = df_id.groupby("Skenario").head(5)
latex_string = df_id_sample.to_latex("tabel_id.tex", index=False, caption=r"Contoh Hasil Pengujian \textit{In-Distribution}", label="tab:id_sample")
df_id_sample.to_csv("tabel_id.csv", index=False)

In [6]:
# Tampilkan 5 baris pertama per skenario sebagai string LaTeX dan CSV
df_ood_sample = df_ood.groupby("Skenario").head(5)
latex_string = df_ood_sample.to_latex("tabel_ood.tex", index=False, caption=r"Contoh Hasil Pengujian \textit{Out-of-Distribution}", label="tab:ood_sample")
df_ood_sample.to_csv("tabel_ood.csv", index=False)